In [6]:
import os
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

from modules.configs.config import Config
from modules.dataset.dataset import SpeakerDataset
from modules.models.ecapa_tdnn import ECAPATDNN
from modules.metrics.metrics import evaluate

def main():
    #Setup Device
    device = Config.DEVICE
    print("=" * 60)
    print(f"Evaluating ECAPA-TDNN on {device}")
    print("=" * 60)
    #Load Dataset
    test_dataset = SpeakerDataset(
        root_dir=Config.TEST_PATH,
        sample_rate=Config.SAMPLE_RATE,
        segment_length=Config.SEGMENT_LENGTH,
        train=False,
        augmentation=False
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=Config.BATCH_SIZE,
        shuffle=False,
        num_workers=Config.NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    print(f"\nTotal Test Speakers : {len(test_dataset.speaker_to_index)}")
    print(f"Total Test Samples  : {len(test_dataset)}")

    # Initialize Model
    model = ECAPATDNN(
        channels=Config.CHANNELS,
        scale=Config.RES2NET_SCALE,
        emb_dim=Config.EMBEDDING_SIZE
    ).to(device)

    # Load Best Checkpoint
    checkpoint_path = os.path.join(Config.CHECKPOINT_DIR, "best_model.pt")
    
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(
            f"Checkpoint not found at: {checkpoint_path}\n"
            "Please train the model first or check the path."
        )

    print(f"\nLoading checkpoint: {checkpoint_path}")
    checkpoint = torch.load(
        checkpoint_path, 
        map_location=device, 
        weights_only=False
    )
    
    model.load_state_dict(checkpoint["model"])
    model.eval()

    # Extract Embeddings
    all_embeddings = []
    all_labels = []

    print("\nExtracting Embeddings...")
    
    with torch.no_grad():
        for features, labels in tqdm(test_loader, desc="Evaluation"):
            
            features = features.to(device, non_blocking=True)
            
            # Forward pass to get 192-dimensional embeddings
            embeddings = model(features)
            
            # Store on CPU 
            all_embeddings.append(embeddings.cpu())
            all_labels.append(labels.cpu())

    # Concatenate all batches into single tensors
    all_embeddings = torch.cat(all_embeddings, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    #Compute Metrics
    metrics = evaluate(all_embeddings, all_labels)
   
    print("\n" + "=" * 60)
    print("EVALUATION RESULTS (Test Set)")
    print("=" * 60)
    
    print(f"Equal Error Rate (EER)   : {metrics['EER'] * 100:.4f}%")
    print(f"Verification Accuracy    : {metrics['Accuracy'] * 100:.2f}%")
    print(f"Minimum DCF (minDCF)     : {metrics['minDCF']:.4f}")
    print(f"False Acceptance Rate    : {metrics['FAR'] * 100:.4f}%")
    print(f"False Rejection Rate     : {metrics['FRR'] * 100:.4f}%")
    
    print("=" * 60)

if __name__ == "__main__":
    main()

Evaluating ECAPA-TDNN on cuda
Scanning Dataset...
./data/LibriSpeech/test-clean
Dataset Loaded
Total Speakers : 40
Total Audio    : 2620

Total Test Speakers : 40
Total Test Samples  : 2620

Loading checkpoint: checkpoints/best_model.pt

Extracting Embeddings...


Evaluation: 100%|██████████| 82/82 [00:10<00:00,  7.56it/s]



EVALUATION RESULTS (Test Set)
Equal Error Rate (EER)   : 6.6234%
Verification Accuracy    : 98.74%
Minimum DCF (minDCF)     : 0.0050
False Acceptance Rate    : 0.0722%
False Rejection Rate     : 44.5026%
